# Maia Lite 0.3.0 — Auditoria qualitativa direcionada SFT v2.2/v2.3

**Objetivo:** investigar problemas de supervisão e de inferência sem refazer o corpus ou treinar. Usa o **mesmo dataset** v2.2 Aya que alimentou o v2.3, compara exemplos do corpus com falhas do benchmark e gera uma amostra estratificada para revisão humana. **Não atribui automaticamente veracidade científica.**

**Execução:** Colab ou máquina local. Se estiver no Colab, monte o Drive. Faça upload dos arquivos `generations(1).csv`, `paired_responses(1).csv` e `run_config(1).json` na pasta `/content/` ou ajuste `BENCH_DIR`. O dataset é lido no Drive sem alterações. A saída fica em pasta separada. Não carrega modelo/GPU.

**Cuidado:** heurísticas são *sinais para inspeção*, não diagnósticos definitivos; taxas de uma amostra enriquecida por risco não representam prevalência do corpus. 

In [ ]:
from pathlib import Path
import json, re, math, hashlib, collections
import pandas as pd
import numpy as np
SEED=20261119
ROOT=Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat')
DATA_DIR=ROOT/'datasets/sft_v2_2_aya'
TRAIN_FILE=DATA_DIR/'sft_v2_2_train.jsonl'
VAL_FILE=DATA_DIR/'sft_v2_2_validation.jsonl'
BENCH_DIR=Path('/content')
OUT_DIR=ROOT/'audits/sft_v2_2_v2_3_qualitative_v1'
SAMPLE_PER_LANG_SPLIT=35
TOP_RISK_PER_LANG_SPLIT=35
# Monte Drive se necessário; a célula não falha fora do Colab.
try:
 from google.colab import drive
 if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
except ImportError: pass
OUT_DIR.mkdir(parents=True,exist_ok=True)
print('Treino:',TRAIN_FILE,' | Validação:',VAL_FILE,' | Saída:',OUT_DIR)

In [ ]:
def load_jsonl(path):
 if not path.exists(): raise FileNotFoundError(f'Arquivo ausente: {path}. Ajuste DATA_DIR, sem reconstruir o dataset.')
 with path.open(encoding='utf-8') as f:
  return [json.loads(line) for line in f if line.strip()]
train=load_jsonl(TRAIN_FILE); val=load_jsonl(VAL_FILE)
assert len(train)==29995 and len(val)==1521, f'Contagens inesperadas: {len(train)}, {len(val)}'
records=[]
for split,rows in [('train',train),('validation',val)]:
 for r in rows:
  records.append({'split':split,'id':str(r.get('id','')),'groupId':str(r.get('groupId','')),
                  'language':str(r.get('language','')).lower(),'source':str(r.get('source',r.get('dataset','unknown'))),
                  'prompt':str(r.get('prompt','')),'completion':str(r.get('completion','')),
                  'license':str(r.get('license',''))})
df=pd.DataFrame(records)
print(df.groupby(['split','language']).size().to_string())
print('Colunas originais (exemplo):',list(train[0]))
assert df['language'].isin(['pt','en','es']).all(), 'Idioma inesperado; inspecione os registros' 

## 1. Sinais de risco e verificações estruturais

Repetição de n-gramas, texto muito curto, respostas que parecem outra pergunta, marcadores de certeza excessiva, números não sustentados pelo prompt, ausência de evidência e possível vazamento entre treino/validação. **Números na resposta não são necessariamente invenção**, e repetição não implica erro científico. 

In [ ]:
def norm(s): return re.sub(r'\s+',' ',str(s).casefold()).strip()
def words(s): return re.findall(r"\b\w+\b",norm(s),flags=re.UNICODE)
def rep_rate(s,n=3):
 w=words(s); grams=list(zip(*(w[i:] for i in range(n)))) if len(w)>=n else []
 return (len(grams)-len(set(grams)))/len(grams) if grams else 0.0
def numeric(s):return set(re.findall(r'(?<!\w)\d+(?:[.,]\d+)?\s*%?',str(s)))
def score_row(r):
 p,a=str(r['prompt']),str(r['completion']); aw=words(a); pw=words(p)
 risk={
  'empty':int(not a.strip()),
  'very_short':int(len(aw)<8),
  'long_answer':int(len(aw)>250),
  'repeated_trigrams':int(rep_rate(a)>=0.18 and len(aw)>=20),
  'starts_as_question':int(a.lstrip().startswith(('¿','?')) or bool(re.match(r'(?i)^\s*(what|why|how|quais|qual|por que|qué|cuáles)\b',a))),
  'prompt_echo':int(len(pw)>=6 and norm(p)[:min(70,len(norm(p)))] in norm(a)),
  'numeric_not_in_prompt':int(bool(numeric(a)-numeric(p))),
  'unqualified_absolute':int(bool(re.search(r'(?i)\b(always|never|guaranteed|sempre|nunca|garantido|siempre|nunca)\b',a))),
 }
 weights={'empty':4,'very_short':2,'long_answer':1,'repeated_trigrams':3,'starts_as_question':2,'prompt_echo':2,'numeric_not_in_prompt':1,'unqualified_absolute':1}
 return pd.Series({**risk,'risk_score':sum(risk[k]*weights[k] for k in risk),'answer_words':len(aw),'trigram_repeat_rate':rep_rate(a)})
flags=df.apply(score_row,axis=1)
df=pd.concat([df,flags],axis=1)
print('Sinais (não equivalem a erros confirmados):')
print(df[list(score_row(df.iloc[0]).index[:-3])].sum().to_string())
print('Distribuição por idioma/split:')
print(df.groupby(['split','language'])[['risk_score','trigram_repeat_rate']].mean().round(3).to_string())

In [ ]:
def digest(s):return hashlib.sha256(norm(s).encode('utf-8')).hexdigest()
train_prompt={digest(x) for x in df.loc[df.split=='train','prompt']}
val_prompt={digest(x) for x in df.loc[df.split=='validation','prompt']}
train_pair={digest(p+'\0'+a) for p,a in zip(df.loc[df.split=='train','prompt'],df.loc[df.split=='train','completion'])}
val_pair={digest(p+'\0'+a) for p,a in zip(df.loc[df.split=='validation','prompt'],df.loc[df.split=='validation','completion'])}
leak={'exact_prompt_overlap_train_val':len(train_prompt & val_prompt),'exact_pair_overlap_train_val':len(train_pair & val_pair),
      'duplicate_ids':int(df.duplicated(['split','id']).sum()),'empty_prompt':int((df.prompt.str.strip()=='').sum()),'empty_completion':int((df.completion.str.strip()=='').sum())}
print(json.dumps(leak,ensure_ascii=False,indent=2))
# Checagem por groupId: se um grupo existe nos dois splits, precisa de revisão.
tr_groups=set(df.loc[(df.split=='train')&(df.groupId!=''),'groupId'])
va_groups=set(df.loc[(df.split=='validation')&(df.groupId!=''),'groupId'])
leak['group_id_overlap_train_val']=len(tr_groups & va_groups)
print('Grupos sobrepostos:',leak['group_id_overlap_train_val'])

## 2. Amostra estratificada e enriquecida por risco

Para cada idioma e split, selecionar exemplos aleatórios e os mais sinalizados, sem duplicá-los. **Revisão cega:** a coluna `risk_score` não aparece no formulário principal para reduzir viés; uma planilha auxiliar registra os sinais. A amostra não estima a prevalência populacional sem ponderação.

In [ ]:
parts=[]
for (split,lang),group in df.groupby(['split','language'],sort=True):
 random_part=group.sample(n=min(SAMPLE_PER_LANG_SPLIT,len(group)),random_state=SEED)
 pool=group.drop(random_part.index)
 high=pool.sort_values(['risk_score','trigram_repeat_rate','id'],ascending=[False,False,True]).head(TOP_RISK_PER_LANG_SPLIT)
 p=pd.concat([random_part.assign(selection='random'),high.assign(selection='risk_enriched')])
 parts.append(p)
sample=pd.concat(parts,ignore_index=True)
sample.insert(0,'review_id',[f'MAIA-{i:04d}' for i in range(1,len(sample)+1)])
review_cols=['review_id','split','language','source','id','groupId','prompt','completion','selection']
review=sample[review_cols].copy()
for col in ['relevance_1_5','factual_accuracy_1_5','instruction_following_1_5','language_quality_1_5','redundancy_1_5','uncertainty_calibration_1_5_or_na','severity_0_3','reviewer_notes','suggested_correction']:
 review[col]=''
review.to_csv(OUT_DIR/'human_review_corpus.csv',index=False,encoding='utf-8-sig')
sample[['review_id','split','language','id','selection','risk_score','trigram_repeat_rate','answer_words','empty','very_short','long_answer','repeated_trigrams','starts_as_question','prompt_echo','numeric_not_in_prompt','unqualified_absolute']].to_csv(OUT_DIR/'risk_signals.csv',index=False)
print('Amostra:',len(review),' | ',review.groupby(['split','language','selection']).size().to_string())

## 3. Falhas observadas no benchmark v2.2 × v2.3

Importe as gerações e produza uma lista de casos prioritários. O benchmark é pequeno (9 perguntas); suas taxas **não devem ser extrapoladas** para todos os temas. Respostas vazias do v2.3 sem `\n` são classificadas como incompatibilidade de formato, não como erro factual.

In [ ]:
def find_bench(name):
 for path in [BENCH_DIR/name,Path('/mnt/data')/name, BENCH_DIR/name.replace('(1)','')]:
  if path.exists():return path
 return None
p=find_bench('generations(1).csv')
if p is None:
 print('Benchmark ausente: faça upload de generations(1).csv em BENCH_DIR. Auditoria do corpus continua válida.')
 benchmark=None
else:
 benchmark=pd.read_csv(p).fillna({'response':''})
 assert {'model','question_id','language','format','strategy','response','ended_eos'}.issubset(benchmark.columns)
 benchmark['response']=benchmark.response.fillna('').astype(str)
 benchmark['empty']=benchmark.response.str.strip().eq('')
 benchmark['risk_repetition']=benchmark.response.map(lambda s:rep_rate(s)>=0.18 and len(words(s))>=20)
 benchmark['risk_numeric']=benchmark.apply(lambda r:bool(numeric(r.response)-numeric(r.prompt)),axis=1)
 benchmark['risk_uncertainty']=benchmark['type'].eq('uncertainty')
 benchmark['priority']=benchmark['empty'].astype(int)*3+benchmark['risk_repetition'].astype(int)*2+benchmark['risk_numeric'].astype(int)+benchmark['risk_uncertainty'].astype(int)*2
 benchmark.sort_values(['priority','language'],ascending=[False,True]).to_csv(OUT_DIR/'benchmark_cases_prioritized.csv',index=False,encoding='utf-8-sig')
 print('Gerações:',len(benchmark))
 print(benchmark.groupby(['model','format']).agg(n=('empty','size'),empty_rate=('empty','mean'),repeat_flag_rate=('risk_repetition','mean')).round(3).to_string())

## 4. Roteiro de revisão humana

Escalas de 1 a 5: **relevância**, **precisão factual**, **seguimento de instruções**, **qualidade linguística**, **redundância** (5 = sem redundância), **calibração da incerteza** (somente quando pertinente). Severidade: 0 = sem problema, 1 = leve, 2 = moderado, 3 = grave. Marcar `NA` quando não aplicável. Verificar afirmações científicas contra fontes independentes, registrando referências em `reviewer_notes`. Não inferir veracidade apenas da fluência. Dois revisores independentes para casos graves ou controversos.

**Critério de decisão v2.4:** primeiro revisar amostra e benchmark; depois definir mudanças mínimas em curadoria/formatos, executar auditoria de contaminação e só então decidir sobre novo SFT. Não usar a própria resposta do modelo como gabarito.

In [ ]:
summary={'audit':'sft_v2_2_v2_3_qualitative_v1','seed':SEED,'train_count':len(train),'validation_count':len(val),
         'review_sample_count':len(review),'random_per_language_split':SAMPLE_PER_LANG_SPLIT,
         'risk_enriched_per_language_split':TOP_RISK_PER_LANG_SPLIT,'integrity':leak,
         'flag_totals':{k:int(df[k].sum()) for k in ['empty','very_short','long_answer','repeated_trigrams','starts_as_question','prompt_echo','numeric_not_in_prompt','unqualified_absolute']},
         'benchmark_rows':len(benchmark) if benchmark is not None else None,
         'status':'AUTOMATED_SCREENING_COMPLETE_HUMAN_REVIEW_PENDING'}
(OUT_DIR/'audit_summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
lines=['# Auditoria direcionada — resultados preliminares','',f"- Corpus: {len(train)} treino, {len(val)} validação.",f"- Amostra para revisão: {len(review)} exemplos.",'- **Revisão factual humana: PENDENTE**. Os sinais automáticos não provam erros.','', '## Integridade']
lines.extend([f'- {k}: {v}' for k,v in leak.items()])
lines.extend(['','## Sinais automáticos (contagem, não erros confirmados)'])
lines.extend([f'- {k}: {v}' for k,v in summary['flag_totals'].items()])
lines.extend(['','## Arquivos','- human_review_corpus.csv — formulário de avaliação','- risk_signals.csv — explicações dos sinais','- benchmark_cases_prioritized.csv — casos do benchmark (se fornecido)','- audit_summary.json — resumo reproduzível','','## Decisão','Não promover nem treinar v2.4 com base apenas nesta triagem. Revisar manualmente os exemplos e documentar evidências.'])
(OUT_DIR/'audit_report.md').write_text(chr(10).join(lines),encoding='utf-8')
print('Arquivos produzidos:')
for f in sorted(OUT_DIR.iterdir()):print(' ',f.name, f.stat().st_size,'bytes')